In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pyspark.sql import SparkSession

In [ ]:
sns.set_style("dark")
plt.style.use("dark_background")

In [ ]:
spark = (
    SparkSession.builder.appName("eda")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.2.0,com.amazonaws:aws-java-sdk-bundle:1.11.375")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.access.key", "test")
    .config("spark.hadoop.fs.s3a.secret.key", "test")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:4566")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .getOrCreate()
)


In [ ]:
# S3 path format
s3_path = "s3a://data/train.csv"

# Set Hadoop configurations for S3 access
hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", "test")
hadoop_conf.set("fs.s3a.secret.key", "test")
hadoop_conf.set("fs.s3a.endpoint", "http://localhost:4566")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")


# Read the file directly into a PySpark DataFrame
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(s3_path)

# Show the first few rows of the DataFrame
df.show()

In [ ]:
# S3 path format
# s3_path = "s3://data/train.csv"

# # Read the file directly into a pandas DataFrame
# df = pd.read_csv(s3_path, storage_options={
#     'key': 'test',
#     'secret': 'test',
#     'client_kwargs': {
#         'endpoint_url': 'http://localhost:4566'
#     }
# })

# df.head()

In [ ]:
# Register the DataFrame as a temporary view
df.createOrReplaceTempView("train")

# Now you can query it with SQL
spark.sql(
    """ --sql
        SELECT * 
        FROM train LIMIT 1;
    """
).show()

In [ ]:
df.columns

In [ ]:
spark.sql("show databases").show()
spark.sql("show tables").show()

In [ ]:
print(spark.sparkContext.uiWebUrl)

In [ ]:
spark.sql(
    """ --sql
        SELECT
            stock_id,
            avg(reference_price) as avg_reference_price,
            min(reference_price) as min_reference_price
        FROM train
        GROUP BY stock_id
        ORDER BY stock_id ASC
        LIMIT 10;
    """
).show()

In [ ]:
spark.sql(
    """ --sql
        SELECT *
        FROM (
            SELECT
                stock_id,
                count(stock_id) as count_stock_id
            FROM train
            GROUP BY stock_id
            ORDER BY count_stock_id DESC
        )
        WHERE count_stock_id > 26455
        LIMIT 10;
    """
).show()